In [1]:
from pathlib import Path
import glob
import re

from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
from rdkit.Chem import rdMolTransforms
from rdkit.Chem import rdMolAlign

In [2]:
# for pdb_path in glob.glob("sampling/train_ref/*.pdb"):
#     pdb_num = int(re.search(r'(\d+).pdb', pdb_path).group(1))
#     mol = Chem.MolFromPDBFile(pdb_path)
#     Chem.MolToPDBFile(mol, f"sampling/test_{pdb_num}.pdb")

In [2]:
def load_pdb(path):
    mol = Chem.MolFromPDBFile(path, removeHs=True, sanitize=False)
    if mol is None:
        print(f"Failed to load {path}")
        return None
    for atom in mol.GetAtoms():
        atom.SetChiralTag(rdchem.ChiralType.CHI_UNSPECIFIED)

    for bond in mol.GetBonds():
        bond.SetStereo(rdchem.BondStereo.STEREONONE)
    try:
        Chem.SanitizeMol(mol)
    except:
        print(f"Failed to sanitize {path}")
        return None
    return mol

def heavy_atom_rmsd(mol_ref, mol_gen):
    # Remove hydrogens for heavy-atom RMSD
    ref = Chem.RemoveHs(mol_ref)
    gen = Chem.RemoveHs(mol_gen)

    return rdMolAlign.GetBestRMS(ref, gen)

def bond_length_mae(mol_ref, mol_gen):
    conf_r = mol_ref.GetConformer()
    conf_g = mol_gen.GetConformer()

    errs = []
    for b in mol_ref.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()

        a_ri = mol_ref.GetAtomWithIdx(i)
        a_rj = mol_ref.GetAtomWithIdx(j)
        a_gi = mol_gen.GetAtomWithIdx(i)
        a_gj = mol_gen.GetAtomWithIdx(j)

        # atom identity checks
        assert a_ri.GetAtomicNum() == a_gi.GetAtomicNum(), f"Atom {i} mismatch"
        assert a_rj.GetAtomicNum() == a_gj.GetAtomicNum(), f"Atom {j} mismatch"

        d_r = conf_r.GetAtomPosition(i).Distance(conf_r.GetAtomPosition(j))
        d_g = conf_g.GetAtomPosition(i).Distance(conf_g.GetAtomPosition(j))

        errs.append(abs(d_r - d_g))

    return float(np.mean(errs))


def angle(a, b, c):
    ba = a - b
    bc = c - b
    norm_ba = np.linalg.norm(ba)
    norm_bc = np.linalg.norm(bc)
    if norm_ba == 0 or norm_bc == 0:
        return 0.0  # Return 0 if vectors are zero (shouldn't happen in valid molecules)
    cos_angle = np.dot(ba, bc) / (norm_ba * norm_bc)
    # Clamp to [-1, 1] to avoid NaN from floating point precision errors
    cos_angle = np.clip(cos_angle, -1.0, 1.0)
    return np.degrees(np.arccos(cos_angle))

def bond_angle_mae(mol_ref, mol_gen):
    conf_r = mol_ref.GetConformer()
    conf_g = mol_gen.GetConformer()

    errs = []
    for atom in mol_ref.GetAtoms():
        nbrs = [n.GetIdx() for n in atom.GetNeighbors()]
        if len(nbrs) < 2:
            continue
        for i in range(len(nbrs)):
            for j in range(i + 1, len(nbrs)):
                a, b, c = nbrs[i], atom.GetIdx(), nbrs[j]
                ar = angle(conf_r.GetAtomPosition(a),
                           conf_r.GetAtomPosition(b),
                           conf_r.GetAtomPosition(c))
                ag = angle(conf_g.GetAtomPosition(a),
                           conf_g.GetAtomPosition(b),
                           conf_g.GetAtomPosition(c))
                errs.append(abs(ar - ag))
    return float(np.mean(errs))

def torsion_mae(mol_ref, mol_gen):
    conf_r = mol_ref.GetConformer()
    conf_g = mol_gen.GetConformer()

    torsions = []

    for bond in mol_ref.GetBonds():
        if bond.IsInRing():
            continue
        if bond.GetBondType() != Chem.BondType.SINGLE:
            continue

        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()

        ai = mol_ref.GetAtomWithIdx(i)
        aj = mol_ref.GetAtomWithIdx(j)

        if ai.GetDegree() < 2 or aj.GetDegree() < 2:
            continue

        ni = sorted(a.GetIdx() for a in ai.GetNeighbors() if a.GetIdx() != j)
        nj = sorted(a.GetIdx() for a in aj.GetNeighbors() if a.GetIdx() != i)

        for a in ni:
            for d in nj:
                tr = rdMolTransforms.GetDihedralDeg(conf_r, a, i, j, d)
                tg = rdMolTransforms.GetDihedralDeg(conf_g, a, i, j, d)

                diff = abs(tr - tg) % 360
                diff = min(diff, 360 - diff)
                torsions.append(diff)

    return float(np.mean(torsions)) if torsions else 0.0


def heavy_atom_rmsf(mol_list, align_first=True):
    """
    Calculate RMSF (Root Mean Square Fluctuation) for heavy atoms across multiple conformations.
    
    Args:
        mol_list: List of RDKit molecule objects (should be the same molecule, different conformations)
        align_first: If True, align all conformations to the first one before calculating RMSF
    
    Returns:
        Average RMSF over all heavy atoms
    """
    if len(mol_list) < 2:
        return 0.0
    
    # Remove hydrogens from all molecules
    mols_noH = [Chem.RemoveHs(mol) for mol in mol_list]
    
    # Check that all molecules have the same number of atoms
    n_atoms = mols_noH[0].GetNumAtoms()
    if not all(mol.GetNumAtoms() == n_atoms for mol in mols_noH):
        print("Warning: Not all conformations have the same number of atoms")
        return None
    
    # Get coordinates for all conformations
    coords_list = []
    for mol in mols_noH:
        conf = mol.GetConformer()
        coords = np.array([conf.GetAtomPosition(i) for i in range(n_atoms)])
        coords_list.append(coords)
    
    # Stack coordinates: [n_confs, n_atoms, 3]
    coords_array = np.stack(coords_list, axis=0)
    
    # Optionally align all conformations to the first one
    if align_first:
        ref_coords = coords_array[0]
        aligned_coords = [ref_coords]  # First conformation is reference
        for i in range(1, coords_array.shape[0]):
            # Align conformation i to reference using Kabsch algorithm
            mol_ref = mols_noH[0]
            mol_to_align = mols_noH[i]
            rmsd = rdMolAlign.GetBestRMS(mol_ref, mol_to_align)
            # Get the aligned conformation
            mol_aligned = Chem.Mol(mol_to_align)
            rdMolAlign.AlignMol(mol_aligned, mol_ref)
            conf_aligned = mol_aligned.GetConformer()
            coords_aligned = np.array([conf_aligned.GetAtomPosition(j) for j in range(n_atoms)])
            aligned_coords.append(coords_aligned)
        coords_array = np.stack(aligned_coords, axis=0)
    
    # Calculate mean position for each atom across conformations
    mean_coords = np.mean(coords_array, axis=0)  # [n_atoms, 3]
    
    # Calculate RMSF for each atom: sqrt(mean((pos_i - mean_pos)^2))
    squared_deviations = np.sum((coords_array - mean_coords[None, :, :]) ** 2, axis=2)  # [n_confs, n_atoms]
    rmsf_per_atom = np.sqrt(np.mean(squared_deviations, axis=0))  # [n_atoms]
    
    # Return average RMSF over all heavy atoms
    return float(np.mean(rmsf_per_atom))



def mmff_energy(mol):
    # AllChem.MMFFOptimizeMolecule(mol)
    props = AllChem.MMFFGetMoleculeProperties(mol)
    ff = AllChem.MMFFGetMoleculeForceField(mol, props)
    return ff.CalcEnergy()


from rdkit.Chem import rdchem
from rdkit.Chem import rdmolops


def clash_count(mol, scale=0.75):
    conf = mol.GetConformer()
    pts = np.array([conf.GetAtomPosition(i) for i in range(mol.GetNumAtoms())])
    vdw = np.array([
        rdchem.GetPeriodicTable().GetRvdw(a.GetAtomicNum())
        for a in mol.GetAtoms()
    ])

    n = mol.GetNumAtoms()
    clashes = 0

    for i in range(n):
        if mol.GetAtomWithIdx(i).GetAtomicNum() == 1:
            continue
        for j in range(i + 1, n):
            if mol.GetAtomWithIdx(j).GetAtomicNum() == 1:
                continue

            path = rdmolops.GetShortestPath(mol, i, j)
            if path is not None and len(path) <= 4:
                continue

            if np.linalg.norm(pts[i] - pts[j]) < scale * (vdw[i] + vdw[j]):
                clashes += 1

    return clashes

In [6]:
import glob
from pathlib import Path
from collections import defaultdict
import numpy as np
from rdkit import Chem
from joblib import Parallel, delayed
import multiprocessing as mp

# ============================================================
# USER SETTINGS
# ============================================================
# mode = "train"
mode = "train"

REF_GLOB = f"sampling/geom_drugs_murcko_{mode}_ref/*.pdb"
PRED_GLOB = f"sampling/geom_identityRot_256_conformer_{mode}_3std_bondlength/samples/*.pdb"

N_JOBS = max(1, mp.cpu_count() // 2)   # change if needed
USE_PARALLEL = True


# ============================================================
# REQUIRED USER FUNCTIONS (already in your codebase)
# ============================================================

# load_pdb(path)
# heavy_atom_rmsd(mol1, mol2)
# heavy_atom_rmsf(mol_list, align_first=True)
# bond_length_mae(mol1, mol2)
# bond_angle_mae(mol1, mol2)
# torsion_mae(mol1, mol2)


# ============================================================
# Utilities
# ============================================================

def canon_smi_from_mol(mol):
    return Chem.CanonSmiles(Chem.MolToSmiles(mol))


def load_and_prepare(pdb_path):
    mol = load_pdb(pdb_path)
    if mol is None:
        return None, None
    mol = Chem.AddHs(mol)
    smi = canon_smi_from_mol(mol)
    return mol, smi


# ============================================================
# Load and Cache Reference Molecules
# ============================================================

print("Loading reference molecules...")

ref_smi_to_mols = defaultdict(list)

for ref_path in glob.glob(REF_GLOB):
    mol, smi = load_and_prepare(ref_path)
    assert mol is not None, f"ref_path mol is None : {ref_path}"
    ref_smi_to_mols[smi].append(mol)

print(f"Loaded {sum(len(v) for v in ref_smi_to_mols.values())} reference conformers")
print(f"{len(ref_smi_to_mols)} unique SMILES in reference")


# ============================================================
# Gather Predicted Molecules
# ============================================================

print("Collecting predicted PDBs...")

pred_paths = [Path(p) for p in glob.glob(PRED_GLOB)]

groups = defaultdict(list)
for p in pred_paths:
    mol_id = p.stem.split("_")[0]
    groups[mol_id].append(p)

print(f"{len(groups)} molecule groups found")


# ============================================================
# Fast Best-Match Metric Computation
# ============================================================

def compute_metrics_for_group(paths):

    mol_confs = []
    best_metrics = None
    valid = 0

    for p in sorted(paths):

        pred_mol, smi = load_and_prepare(p)
        if pred_mol is None:
            continue

        ref_mols = ref_smi_to_mols.get(smi)
        if ref_mols is None:
            continue

        # ---- Compute metrics vs all refs, keep best RMSD ----
        rmsds = []
        bl = []
        ba = []
        tor = []

        for ref in ref_mols:
            rmsds.append(heavy_atom_rmsd(pred_mol, ref))
            bl.append(bond_length_mae(pred_mol, ref))
            ba.append(bond_angle_mae(pred_mol, ref))
            tor.append(torsion_mae(pred_mol, ref))

        if not rmsds:
            continue

        best_idx = int(np.argmin(rmsds))
        metrics = (
            rmsds[best_idx],
            bl[best_idx],
            ba[best_idx],
            tor[best_idx],
        )

        mol_confs.append(pred_mol)
        valid += 1

        if best_metrics is None or metrics[0] < best_metrics[0]:
            best_metrics = metrics

    # ---- RMSF across conformers ----
    rmsf_val = None
    if len(mol_confs) >= 2:
        rmsf_val = heavy_atom_rmsf(mol_confs, align_first=True)

    return best_metrics, rmsf_val, valid


# ============================================================
# Run Computation (Parallel or Serial)
# ============================================================

print("Running metric computation...")

if USE_PARALLEL:
    results = Parallel(n_jobs=N_JOBS, backend="loky")(
        delayed(compute_metrics_for_group)(paths)
        for paths in groups.values()
    )
else:
    results = [compute_metrics_for_group(paths) for paths in groups.values()]


# ============================================================
# Aggregate Results
# ============================================================

heavy_atom_min_rmsd = []
bond_length_min_mae = []
bond_angle_min_mae = []
torsion_min_mae = []
rmsf_list = []

valid_mols = 0

for best_metrics, rmsf_val, valid in results:

    valid_mols += valid

    if best_metrics is not None:
        heavy_atom_min_rmsd.append(best_metrics[0])
        bond_length_min_mae.append(best_metrics[1])
        bond_angle_min_mae.append(best_metrics[2])
        torsion_min_mae.append(best_metrics[3])

    if rmsf_val is not None:
        rmsf_list.append(rmsf_val)


# ============================================================
# Summary
# ============================================================

print("\n===== SUMMARY =====")
print("Valid molecules:", valid_mols)
print("Num RMSD entries:", len(heavy_atom_min_rmsd))
print("Num RMSF entries:", len(rmsf_list))

print("Mean Heavy Atom RMSD:", np.mean(heavy_atom_min_rmsd))
print("Mean Bond Length MAE:", np.mean(bond_length_min_mae))
print("Mean Bond Angle MAE:", np.mean(bond_angle_min_mae))
print("Mean Torsion MAE:", np.mean(torsion_min_mae))
print("Mean RMSF:", np.mean(rmsf_list))


Loading reference molecules...
Loaded 10767 reference conformers
60 unique SMILES in reference
30 molecule groups found
Running metric computation...


[16:11:59] Explicit valence for atom # 0 O, 3, is greater than permitted


Failed to sanitize sampling/geom_identityRot_256_conformer_train_3std_bondlength/samples/373497472b228cf5bb42f6c4a30e8dd1da6e251a862f8e22743a374c5cd55f3b_3.pdb

===== SUMMARY =====
Valid molecules: 119
Num RMSD entries: 24
Num RMSF entries: 24
Mean Heavy Atom RMSD: 1.0393848794506508
Mean Bond Length MAE: 0.44413973551042457
Mean Bond Angle MAE: 14.142490129181231
Mean Torsion MAE: 44.12479916477388
Mean RMSF: 3.033199153611872


In [7]:
# train - all
print(np.mean(heavy_atom_min_rmsd_list), len(heavy_atom_min_rmsd_list))
print(np.mean(bond_length_min_mae_list))
print(np.mean(bond_angle_min_mae_list))
print(np.mean(torsion_min_mae_list))
print(np.mean(rmsf_list), len(rmsf_list))
print(valid_mols)
# print(np.mean(energy_list), len(energy_list))

0.9267492452212903 29
0.545315313830162
15.708515700056578
61.18656122378276
2.7537358466653465 29
142


In [9]:
print(np.mean(heavy_atom_min_rmsd_list), len(heavy_atom_min_rmsd_list))
print(np.mean(bond_length_min_mae_list))
print(np.mean(bond_angle_min_mae_list))
print(np.mean(torsion_min_mae_list))
print(np.mean(rmsf_list), len(rmsf_list))
print(valid_mols)

nan 0
nan
nan
nan
nan 0
0


/var/local/dy4652/miniconda3/envs/proteinzen/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/var/local/dy4652/miniconda3/envs/proteinzen/lib/python3.10/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
